In [ ]:
import numpy as np
from vpython import sphere, arrow, vector, color, canvas, graph, gcurve, rate, mag

In [ ]:
# ==========================================
# PARÁMETROS GENERALES (Tabla 1)
# ==========================================
k = 1.0
Q = 1.0
R = 1.0
N_default = 60
m = 1.0
dt_default = 1e-3

In [ ]:
# ==========================================
# FUNCIONES DE CAMPO Y POTENCIAL
# ==========================================
def posiciones_anillo(N, R_val=R):
    """Devuelve un arreglo numpy con las posiciones r_j de las N cargas."""
    j = np.arange(N)
    phi_j = 2 * np.pi * j / N
    pos = np.zeros((N, 3))
    pos[:, 0] = R_val * np.cos(phi_j)
    pos[:, 1] = R_val * np.sin(phi_j)
    pos[:, 2] = 0.0
    return pos

def campo_electrico(r_vec, N=N_default, Q_val=Q, R_val=R):
    """Calcula E(r) numéricamente según la Ec. (2) usando superposición."""
    r = np.array(r_vec)
    r_j = posiciones_anillo(N, R_val)
    q_j = Q_val / N
    
    diff = r - r_j  # Vectores r - r_j
    dists = np.linalg.norm(diff, axis=1)
    
    # Prevenir división por cero si coincide con una carga puntual
    dists_cubed = np.where(dists == 0, np.inf, dists**3)
    
    E_contribs = k * q_j * diff / dists_cubed[:, np.newaxis]
    E_tot = np.sum(E_contribs, axis=0)
    return vector(E_tot[0], E_tot[1], E_tot[2])

def E_z_analitico(z, Q_val=Q, R_val=R):
    """Ecuación (3): Campo analítico sobre el eje z."""
    return (k * Q_val * z) / ((R_val**2 + z**2)**(1.5))

def V_analitico(z, Q_val=Q, R_val=R):
    """Ecuación (4): Potencial analítico sobre el eje z."""
    return (k * Q_val) / np.sqrt(R_val**2 + z**2)

In [ ]:
# ==========================================
# SECCIÓN 1: CAMPO ELÉCTRICO DEL ANILLO
# ==========================================
print("=== SECCIÓN 1: CAMPO ELÉCTRICO DEL ANILLO ===")

# 1.1 Evaluación de E_z en (0, 0, z) y error relativo
puntos_z = [0.5, 1.0, 2.0]
print("\n--- Punto 1.1: Comparación E_z numérico vs analítico ---")
for z in puntos_z:
    E_num = campo_electrico([0, 0, z]).z
    E_ana = E_z_analitico(z)
    err_rel = abs(E_num - E_ana) / abs(E_ana)
    print(f"z = {z:3.1f} | E_num = {E_num:.6f} | E_ana = {E_ana:.6f} | Error Relativo = {err_rel:.2e}")

In [ ]:
# 1.2 Campo en punto cercano (0.8, 0, 0.1) variando N
N_valores = [10, 30, 60, 120]
r_cercano = [0.8, 0.0, 0.1]
print("\n--- Punto 1.2: Campo en (0.8, 0, 0.1) para varios N ---")
print(f"{'N':<5} | {'Ex':<10} | {'Ey':<10} | {'Ez':<10} | {'|E|':<10}")
print("-" * 55)
for N_i in N_valores:
    E_i = campo_electrico(r_cercano, N=N_i)
    print(f"{N_i:<5} | {E_i.x:<10.6f} | {E_i.y:<10.6f} | {E_i.z:<10.6f} | {mag(E_i):<10.6f}")

In [ ]:
# 1.3 Construcción de la escena gráfica en VPython
scene1 = canvas(title="1.3 Visualización del Campo Eléctrico del Anillo", width=800, height=600)

# Dibujar N cargas del anillo
r_j_coords = posiciones_anillo(N_default)
for pos in r_j_coords:
    sphere(pos=vector(pos[0], pos[1], pos[2]), radius=0.04, color=color.red)

# Rejilla 1: plano y = 0
grid_coords = np.arange(-2.0, 2.1, 0.4)
for x in grid_coords:
    for z in grid_coords:
        r_pt = vector(x, 0, z)
        # Omitir si está a menos de 0.15 de alguna carga
        if min(np.linalg.norm(np.array([x, 0, z]) - r_j_coords, axis=1)) >= 0.15:
            E_v = campo_electrico([x, 0, z])
            E_mag = mag(E_v)
            if E_mag > 0:
                arrow(pos=r_pt, axis=0.3 * (E_v / E_mag), color=color.cyan, shaftwidth=0.01)

# Rejilla 2: plano x = 0
for y in grid_coords:
    for z in grid_coords:
        if y == 0: continue # Evitar duplicar la línea central
        r_pt = vector(0, y, z)
        if min(np.linalg.norm(np.array([0, y, z]) - r_j_coords, axis=1)) >= 0.15:
            E_v = campo_electrico([0, y, z])
            E_mag = mag(E_v)
            if E_mag > 0:
                arrow(pos=r_pt, axis=0.3 * (E_v / E_mag), color=color.cyan, shaftwidth=0.01)

In [ ]:
# ==========================================
# SECCIÓN 2: OSCILACIÓN DE UNA CARGA NEGATIVA SOBRE EL EJE
# ==========================================
def simular_oscilacion(r0, q=-1.0, dt=1e-3, t_max=30.0, detener_por_cercania=False):
    """Función para integrar el movimiento con Euler y graficar energías/posiciones."""
    scene = canvas(title=f"Simulación Carga Negativa r0={r0}, dt={dt}", width=800, height=500)
    
    # Anillo
    r_j_coords = posiciones_anillo(N_default)
    for pos in r_j_coords:
        sphere(pos=vector(pos[0], pos[1], pos[2]), radius=0.04, color=color.red)
        
    # Partícula
    particula = sphere(pos=r0, radius=0.05, color=color.blue if q < 0 else color.orange, make_trail=True)
    v = vector(0, 0, 0)
    
    # Gráficos de energía y posición
    g = graph(title="Energía y Posición vs Tiempo", xtitle="t", ytitle="Valores", width=800, height=300)
    c_z = gcurve(color=color.black, label="z(t)")
    c_K = gcurve(color=color.blue, label="K(t)")
    c_U = gcurve(color=color.green, label="U(t)")
    c_Etot = gcurve(color=color.red, label="E_tot(t)")
    
    t = 0.0
    z_prev = r0.z
    cruces_z = []
    
    while t < t_max:
        rate(500)
        
        # Fuerza y Euler
        E_v = campo_electrico([particula.pos.x, particula.pos.y, particula.pos.z])
        F_v = q * E_v
        
        v = v + (F_v / m) * dt
        particula.pos = particula.pos + v * dt
        
        # Mediciones de energía (si está sobre el eje x=0, y=0)
        v_mag2 = mag(v)**2
        K = 0.5 * m * v_mag2
        
        # Si está sobre el eje, calculamos U analítico
        if abs(particula.pos.x) < 1e-5 and abs(particula.pos.y) < 1e-5:
            U = q * V_analitico(particula.pos.z)
            Etot = K + U
            c_U.plot(t, U)
            c_Etot.plot(t, Etot)
        
        c_z.plot(t, particula.pos.z)
        c_K.plot(t, K)
        
        # Detección de paso de z positivo a negativo (para medir período)
        if z_prev > 0 and particula.pos.z <= 0:
            cruces_z.append(t)
        z_prev = particula.pos.z
        
        # Condición de parada por cercanía al anillo
        if detener_por_cercania:
            dist_min = min(np.linalg.norm(np.array([particula.pos.x, particula.pos.y, particula.pos.z]) - r_j_coords, axis=1))
            if dist_min < 0.05:
                print(f"Simulación detenida en t={t:.3f} por colisión/cercanía (<0.05) a una carga.")
                break
                
        t += dt
        
    # Cálculo del período promedio si hubo al menos dos cruces
    if len(cruces_z) >= 2:
        periodo_medido = 2 * (cruces_z[1] - cruces_z[0])
        print(f"Período medido (r0={r0}): T_medido ≈ {periodo_medido:.4f}")
    
    return K + (q * V_analitico(particula.pos.z) if abs(particula.pos.x)<1e-5 else 0)

In [ ]:
print("\n=== EJECUTANDO SIMULACIONES DE LA SECCIÓN 2 ===")

# 2.1 Libere la partícula desde r0 = (0, 0, 0.2), t_max = 30
print("\n--- Punto 2.1: r0 = (0, 0, 0.2), dt = 1e-3 ---")
E_tot_final_1 = simular_oscilacion(vector(0, 0, 0.2), q=-1.0, dt=1e-3, t_max=30.0)

In [ ]:
# 2.2 Repita con r0 = (0, 0, 1.5)
print("\n--- Punto 2.2: r0 = (0, 0, 1.5), dt = 1e-3 ---")
E_tot_final_2 = simular_oscilacion(vector(0, 0, 1.5), q=-1.0, dt=1e-3, t_max=30.0)

In [ ]:
# 2.3 Repita r0 = (0, 0, 1.5) con dt = 1e-2 y comparar energías
print("\n--- Punto 2.3: r0 = (0, 0, 1.5), dt = 1e-2 (Comparación de Energía) ---")
E_tot_final_3 = simular_oscilacion(vector(0, 0, 1.5), q=-1.0, dt=1e-2, t_max=30.0)

In [ ]:
# 2.4 Perturbación r0 = (0.01, 0, 0.2) con t_max = 10 y detección de cercanía
print("\n--- Punto 2.4: Perturbación fuera del eje r0 = (0.01, 0, 0.2) ---")